# Lesson 25 — Context Managers: Practice (SUPPORTING)

Work in `lessons/lesson-25-context-managers/` (point a terminal at that folder).

Before you start, read `lesson.md` — every construct used in these exercises is
taught there. Write each deliverable in its stated file (R16 — no
`solution.py`, no double-write), then run each **Driver:** cell. Every
`Expected` block must match byte-exact.

You create: `guards.py` (E1–M2) · `release.py` + `publish.py` (Challenge,
which imports `guards.py` — write once, reuse).
Provided: `buggy_publish.py` (buggy — true hunt, you fix it in place).
Installs: none — stdlib only (`contextlib` + `Path.replace` taught in lesson.md).


## E1 — Time a block with a class

## Intent
**Record wall time of any block — even one that raises — without hiding the error.**

**Type:** REAL-WORLD — Easy — writes 1 file

**Task**
In `guards.py`, define the `Timer` class: `__enter__` stamps
`perf_counter` and returns `self`; `__exit__` records `elapsed` and
returns falsy so whatever the block raised keeps propagating. Timing
observes; it never interferes.

**File tree**

- `guards.py` (`Timer`)

**Interface contract**

- `with Timer() as t:` exposes `t.elapsed: float` afterwards, on clean
  and raising exits alike.

**Constraints**

- **`__exit__` must not suppress (falsy return, always).**
- **Elapsed is measured, not defaulted** (driver sleeps, then checks).

**Coverage**

- Class protocol → lesson.md Core Concepts 1–2.

**Expected** (Driver cell):
```text
E1 OK
```


In [ ]:
import time
import guards as g

with g.Timer() as t:
    time.sleep(0.05)
assert isinstance(t.elapsed, float) and t.elapsed >= 0.04, t.elapsed
try:
    with g.Timer() as t2:
        raise RuntimeError("x")
except RuntimeError:
    pass
else:
    raise AssertionError("swallowed")
assert isinstance(t2.elapsed, float) and t2.elapsed >= 0.0
print("E1 OK")

## E2 — The same timer as a generator

## Intent
**Rebuild the E1 guard in five lines — identical contract, no dunders.**

**Type:** REAL-WORLD — Easy — writes 1 file

**Task**
In `guards.py`, define the `@contextmanager` generator `timed()`.
Yield exactly one record dict `{"elapsed": 0.0}`; set the elapsed in a
`finally` so throw-ins still record. Same contract as E1, authored the
L24 way.

**File tree**

- `guards.py` (`timed`)

**Interface contract**

- `with timed() as rec:` exposes `rec["elapsed"]: float` afterwards, on
  clean and raising exits alike.

**Constraints**

- **Exactly one `yield`, teardown in `finally`.**
- **`contextlib.contextmanager` is the only new import (taught in
  lesson.md Core Concept 4).**

**Coverage**

- Generator CM → lesson.md Core Concept 4.

**Expected** (Driver cell):
```text
E2 OK
```


In [ ]:
import time
import importlib
import guards as g; importlib.reload(g)

with g.timed() as rec:
    time.sleep(0.05)
assert isinstance(rec["elapsed"], float) and rec["elapsed"] >= 0.04
try:
    with g.timed() as rec2:
        time.sleep(0.02)
        raise RuntimeError("x")
except RuntimeError:
    pass
else:
    raise AssertionError("swallowed")
assert rec2["elapsed"] >= 0.015, rec2
print("E2 OK")

## E3 — Prove exit runs on every path

## Intent
**Show the guarantee holds on clean, raising, returning, and breaking exits — plus a missing-file edge.**

**Type:** REAL-WORLD — Easy — writes 1 file

**Task**
In `guards.py`, define the `Trail(events)` class (`__enter__` appends
`"enter"` and returns `self`; `__exit__` appends `"exit"`, propagates)
and `read_first(path)` (first line stripped via `with`-open; empty file
yields `""`; missing file propagates `FileNotFoundError` — never
swallowed, never defaulted).

**File tree**

- `guards.py` (`Trail`, `read_first`)

**Interface contract**

- Every `with Trail(events):` block appends exactly
  `["enter", "exit"]` — clean, raising (after catch outside), early
  `return`, loop `break` alike.
- `read_first(path)` returns stripped first line, `""` when empty.

**Constraints**

- **`__exit__` takes the triple and propagates (falsy return).**
- **Missing file raises — no invented defaults, no `None` returns.**

**Coverage**

- Exit guarantee → lesson.md Core Concept 2 (+ L8 finally retrieval).

**Expected** (Driver cell):
```text
E3 OK
```


In [ ]:
from pathlib import Path
import tempfile
import importlib
import guards as g; importlib.reload(g)

events = []
with g.Trail(events):
    pass
assert events == ["enter", "exit"], events
events.clear()
try:
    with g.Trail(events):
        raise ValueError("x")
except ValueError:
    pass
assert events == ["enter", "exit"], events
events.clear()
def f():
    with g.Trail(events):
        return 1
assert f() == 1 and events == ["enter", "exit"], events
events.clear()
for _ in [1]:
    with g.Trail(events):
        break
assert events == ["enter", "exit"], events
with tempfile.TemporaryDirectory() as td:
    p = Path(td) / "a.txt"
    p.write_text("  hi  \nsecond\n", encoding="utf-8")
    assert g.read_first(p) == "hi"
    e = Path(td) / "empty.txt"
    e.write_text("", encoding="utf-8")
    assert g.read_first(e) == ""
    try:
        g.read_first(Path(td) / "nope.txt")
    except FileNotFoundError:
        pass
    else:
        raise AssertionError("swallowed")
print("E3 OK")

## M1 — Publish atomically or not at all

## Intent
**Write reports that are whole or absent — a half-file must be unproducible.**

**Type:** REAL-WORLD — Medium — writes 1 file

**Task**
In `guards.py`, define the `AtomicWriter(target)` class. `__init__`
stores `Path(target)`; the temp sibling is `target.with_suffix(".tmp")`.
`__enter__` opens the temp for writing (`"w"`, `encoding="utf-8"`) and
returns the open handle. `__exit__` closes, then on clean exit replaces
the target (`Path.replace` — atomic on one filesystem); on error it
demolishes the temp (`unlink(missing_ok=True)`) and propagates. The
target is never truncated before commit.

**File tree**

- `guards.py` (`AtomicWriter`)

**Interface contract**

- Clean block → target holds exactly the written bytes; no temp
  sibling remains.
- Raising block → target byte-identical to before; no temp sibling
  remains; the error propagates.

**Constraints**

- **Temp rule is exactly `target.with_suffix(".tmp")`.**
- **Target never truncated before commit (all writes go to temp).**
- **Discard + propagate on error (falsy return, always).**
- **`Path.replace` / `unlink(missing_ok=True)` as taught (first use).**

**Coverage**

- Atomic publish → lesson.md Core Concept 5 (+ discard mechanics).

**Expected** (Driver cell):
```text
M1 OK
```


In [ ]:
import importlib
from pathlib import Path
import tempfile
import guards as g; importlib.reload(g)

with tempfile.TemporaryDirectory() as td:
    td = Path(td)
    tgt = td / "report.txt"
    with g.AtomicWriter(tgt) as h:
        h.write("hello\n")
    assert tgt.read_text(encoding="utf-8") == "hello\n"
    assert sorted(p.name for p in td.iterdir()) == ["report.txt"]
    tgt.write_text("old\n", encoding="utf-8")
    try:
        with g.AtomicWriter(tgt) as h2:
            h2.write("new\n")
            raise RuntimeError("boom")
    except RuntimeError:
        pass
    else:
        raise AssertionError("swallowed")
    assert tgt.read_text(encoding="utf-8") == "old\n"
    assert sorted(p.name for p in td.iterdir()) == ["report.txt"]
print("M1 OK")

## M2 — Compose a timed publish yourself

## Intent
**Wire your timer around your publisher — failed publishes leave no trace.**

**Type:** REAL-WORLD — Medium — writes 1 file

**Task**
In `guards.py`, define `timed_publish(lines, target)` from YOUR `Timer`
+ `AtomicWriter` (this composition is the exercise — the driver checks
behavior only). Validate each line (blank-after-strip → `ValueError`
with exact message below); write canonical `text + "\n"` per line;
return the elapsed float. Small driver data may collect-then-write;
say so in one comment (streaming posture is graded at the Challenge).

**File tree**

- `guards.py` (`timed_publish` — no new file)

**Interface contract**

- Returns elapsed `float`; target holds canonical lines on success;
  target untouched (absent, if new) on validation failure.

**Constraints**

- **`ValueError("blank line in draft")` on the first blank line.**
- **Reuse your own guards (no reimplemented timing, no second
  publisher).**
- **Canonicalize at write (`text + "\n"`); never write raw lines.**

**Coverage**

- Learner-authored composition → M2 law + Core Concepts 1/5.

**Expected** (Driver cell):
```text
M2 OK
```


In [ ]:
from pathlib import Path
import tempfile
import importlib
import guards as g; importlib.reload(g)

with tempfile.TemporaryDirectory() as td:
    td = Path(td)
    tgt = td / "out.txt"
    el = g.timed_publish(["a\n", "b"], tgt)
    assert isinstance(el, float) and el >= 0.0
    assert tgt.read_text(encoding="utf-8") == "a\nb\n"
    bad = td / "bad.txt"
    try:
        g.timed_publish(["ok\n", "   \n"], bad)
    except ValueError as e:
        assert str(e) == "blank line in draft"
    else:
        raise AssertionError("no raise")
    assert not bad.exists()
print("M2 OK")

## H1 — Hunt: four guard bugs in buggy_publish

## Intent
**Diagnose four real context-manager failures in a provided module — fix each where it lives.**

**Type:** DEBUG — Hard — fixes 1 file

**Task**
`buggy_publish.py` is provided, buggy: four planted failures, one per
Pitfall (swallowed error, leaked temp, wrong `as`-target, bare yield).
Nothing else in the file is wrong. Run the DEBUGGING protocol:
reproduce via the driver, read the traceback, locate, hypothesize, fix,
explain. Fix in place; do not restructure the module.

**File tree**

- `buggy_publish.py` (`Timer`, `AtomicWriter`, `Opener`, `timed` —
  YOU fix all four)

**Interface contract** (fixed states, exact)

- `Timer` propagates block errors (elapsed still recorded).
- `AtomicWriter` leaves pre-existing targets byte-identical on error,
  with no temp sibling remaining.
- `Opener` hands the body a writable handle.
- `timed` records elapsed even when the block raises.

**Constraints**

- **Four fixes only — no refactoring, no new symbols.**
- **Each fix stays in its own layer.**
- **Fill one 7-row chain per bug in the next cell, as you fix.**

**Coverage**

- The four guard failures → lesson.md Pitfalls 1–4.

**Expected** (Driver cell):
```text
H1 OK
```


### H1 chain reports (one per bug — fill as you fix)

#### Bug 1 — Timer swallows
| Step | Content |
|---|---|
| Symptom | AssertionError ---> 14     raise AssertionError("swallowed") |
| Trigger | with bp.Timer() |
| Hypothesis | the Timer exit swallow exception |
| Evidence | the raised ValueError after calling the Timer wasnt catched in the exception block |
| Cause | return True in \_\_exit\_\_() |
| Fix | return False |
| Proof | passed |

#### Bug 2 — AtomicWriter leaks
| Step | Content |
|---|---|
| Symptom | AssertionError ---> 30 assert tgt.read_text(encoding="utf-8") == "old\n" |
| Trigger | with tempfile.TemporaryDirectory() as td: |
| Hypothesis | the unlink temp path override written file |
| Evidence | 'new\n' |
| Cause | \_\_exit\_\_ didnt unlink temp path|
| Fix | add unlink |
| Proof | passed: 'old\n' |

#### Bug 3 — Opener target
| Step | Content |
|---|---|
| Symptom | AttributeError: 'NoneType' object has no attribute 'write' |
| Trigger | with bp.Opener(p2) as h2: |
| Hypothesis | \_\_enter\_\_ didnt return self._handle|
| Evidence | no return on \_\_enter\_\_ |
| Cause | no return on \_\_enter\_\_  |
| Fix | add return self._handle |
| Proof | passed |

#### Bug 4 — timed teardown
| Step | Content |
|---|---|
| Symptom | AssertionError: {'elapsed': 0.0} |
| Trigger | ---> 43     assert rec["elapsed"] >= 0.015, rec |
| Hypothesis | raised RuntimeError caused timer not counting |
| Evidence | 0.0 |
| Cause | no exception block to try yield and finally count time |
| Fix | add exception block |
| Proof | passed |


In [ ]:
import importlib
import time
from pathlib import Path
import tempfile
import buggy_publish as bp
importlib.reload(bp)

try:
    with bp.Timer():
        raise ValueError("x")
except ValueError:
    pass
else:
    raise AssertionError("swallowed")
with bp.Timer() as t:
    pass
assert isinstance(t.elapsed, float) and t.elapsed >= 0.0
with tempfile.TemporaryDirectory() as td:
    td = Path(td)
    tgt = td / "r.txt"
    tgt.write_text("old\n", encoding="utf-8")
    try:
        with bp.AtomicWriter(tgt) as h:
            h.write("new\n")
            raise RuntimeError("boom")
    except RuntimeError:
        pass
    else:
        raise AssertionError("swallowed")
    assert tgt.read_text(encoding="utf-8") == "old\n"
    assert sorted(p.name for p in td.iterdir()) == ["r.txt"]
    p2 = td / "w.txt"
    with bp.Opener(p2) as h2:
        h2.write("x\n")
    
    assert p2.read_text(encoding="utf-8") == "x\n"
    try:
        with bp.timed() as rec:
            time.sleep(0.02)
            raise RuntimeError("boom")
    except RuntimeError:
        pass
    assert rec["elapsed"] >= 0.015, rec
print("H1 OK")

## Challenge — Dawn publish run for the report

## Intent
**Ship a publish run that aborts loudly on poison and never leaves a half-file.**

**Type:** REAL-WORLD — Micro Build — writes 2 files

**Task**
The publisher runs before dawn: `python publish.py <draft.txt>
<report.txt>`. It validates each draft line, publishes `line N: text`
entries plus loud `skip` lines into the report, and aborts the whole
run on a poison line — leaving a pre-existing target byte-identical.

Create `release.py` and `publish.py` in this folder. Reuse `guards.py`
(`AtomicWriter` — write once, import, never double-write).

**Flow** (the whole wiring — every arrow is required)

```text
main reads <draft.txt> (always a valid text file; small draft, read whole)
  → per (i, raw) in file order:
      stripped = raw.strip()
      "POISON" → abort: stderr + exit 1, target untouched
      blank → skip line (invalid line: empty)
      else → numbered line
  → text = build_release(lines)
  → AtomicWriter(target): write text (discard mechanics graded at M1)
  → print the same text to stdout
```

The draft is always a valid text file — no corrupt-file handling
required. "Missing draft" means the path does not exist or is not a
file. Abort-before-write keeps pre-existing targets byte-identical;
discard mechanics are M1-graded (stated division, no overclaim).

**File tree**

- `release.py` (`build_release`)
- `publish.py` (`main`)

**Interface contract**

- `release.build_release(lines)` returns header + lines + trailing
  newline (header alone when empty). It performs no I/O.
- Header shape: `published P, skipped S` (counts derived from line
  prefixes `line ` / `skip `).
- `publish.main(argv)` — house shape (L20–L24): full `sys.argv`
  (`main(sys.argv)` guard), paths at `argv[1]` / `argv[2]`, `-> None`,
  errors leave via `sys.exit(1)` after one stderr line.

**Constraints**

- **Per line: poison-check → blank-check → numbered line, in file
  order.**
- **Poison prints `poison draft` to stderr and exits 1 — target
  byte-identical to before (or absent, if new).**
- **Wrong argc prints `usage: publish.py <draft.txt> <report.txt>`
  to stderr and exits 1.**
- **Missing draft prints `bad draft: not found` to stderr and exits 1.**
- **Write ONLY through `AtomicWriter` (never raw-open the target).**
- **Stdout and the report file are byte-identical.**

**Coverage**

- Guard composition + atomic publish → lesson.md Core Concepts 1–6 +
  house entry (L20–L24 retrieval).

**Expected** (Driver cell, real subprocess runs, fixtures of 3 + poison):
```text
CH OK
```


In [ ]:
import subprocess, sys, tempfile, pathlib
HOME = pathlib.Path.cwd()
draft_lines = ["alpha", "", "beta"]
poison_lines = ["alpha", "POISON", "beta"]
expected = ("published 2, skipped 1\n" "line 0: alpha\n"
    "skip 1: invalid line: empty\n" "line 2: beta\n")
with tempfile.TemporaryDirectory() as td:
    td = pathlib.Path(td)
    df, rp = td / "draft.txt", td / "report.txt"
    df.write_text("\n".join(draft_lines) + "\n", encoding="utf-8")
    run = subprocess.run(
        [sys.executable, "publish.py", str(df), str(rp)],
        capture_output=True, text=True, cwd=HOME)
    assert run.returncode == 0, run.stderr
    assert run.stdout == expected, repr(run.stdout)
    assert rp.read_text(encoding="utf-8") == expected
    rp.write_text("old report\n", encoding="utf-8")
    pf = td / "poison.txt"
    pf.write_text("\n".join(poison_lines) + "\n", encoding="utf-8")
    bad = subprocess.run(
        [sys.executable, "publish.py", str(pf), str(rp)],
        capture_output=True, text=True, cwd=HOME)
    assert bad.returncode == 1
    assert bad.stderr == "poison draft\n"
    assert rp.read_text(encoding="utf-8") == "old report\n"
    assert sorted(p.name for p in td.iterdir()) == sorted(
        [df.name, rp.name, pf.name])
    fresh = td / "fresh.txt"
    bad2 = subprocess.run(
        [sys.executable, "publish.py", str(pf), str(fresh)],
        capture_output=True, text=True, cwd=HOME)
    assert bad2.returncode == 1
    assert not fresh.exists()
    usage = subprocess.run(
        [sys.executable, "publish.py", str(df)],
        capture_output=True, text=True, cwd=HOME)
    assert usage.returncode == 1
    assert usage.stderr == "usage: publish.py <draft.txt> <report.txt>\n"
    missing = subprocess.run(
        [sys.executable, "publish.py", str(td / "nope.txt"), str(rp)],
        capture_output=True, text=True, cwd=HOME)
    assert missing.returncode == 1
    assert missing.stderr == "bad draft: not found\n"
print("CH OK")